note to self: running this on conda

In [37]:
import numpy as np
import random
import math

Building the dataset with a sherlock holmes book as the source.

In [88]:
# downloading text
import urllib.request
import re

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

text = open("shakespeare.txt").read().lower()

# clean up extra blank lines
text = re.sub(r'\n\s*\n', '\n', text)

# split by newlines (shakespeare has one line per speaker)
lines = text.split('\n')
lines = [line.strip() for line in lines if line.strip()]

# join back and split on punctuation
text = ' '.join(lines)
sentences = re.split(r'[.!?]+', text)
sentences = [s.strip().split() for s in sentences if len(s.strip().split()) > 3]

# building word to int mapping
vocab = sorted(list(set(word for sentence in sentences for word in sentence)))
wtoi = {w:i+1 for i,w in enumerate(vocab)}
wtoi['<S>'] = 0
itow = {i:w for w,i in wtoi.items()}
vocab_size = len(itow)
print(vocab_size)

# building the actual dataset
context_size = 4
def build_dataset(sentences):
    X, Y = [], []

    for sentence in sentences:
        context = [0] * context_size
        for word in sentence:
            ix = wtoi[word]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    return np.array(X), np.array(Y)

random.shuffle(sentences)
n1 = int(0.8*len(sentences))
n2 = int(0.9*len(sentences))
Xtr, Ytr = build_dataset(sentences[:n1])
Xdev, Ydev = build_dataset(sentences[n1:n2])
Xte, Yte = build_dataset(sentences[n2:])

19598


In [89]:
Xtr.shape, Ytr.shape

((159799, 4), (159799,))

now that i have the datasets in X and Y, i'll start building the model!
* my goal is to be able to customize the number of dimensions/layers/neurons in the model, so i'll build a class for the model

In [93]:
class makemore:
    def __init__(self, dimensions, context_size, vocablen, batch_size, lr=0.1):
        self.dim = dimensions
        self.context_len = context_size
        self.vocab_size = vocablen
        self.layers = []
        self.batch_size = batch_size
        self.lr=lr

    def matmul(self, h, W, b = None):
        # c = np.zeros((h.shape[0], W.shape[1]))
        # for i in range(h.shape[0]):
        #     for k in range(W.shape[1]):
        #         for j in range(h.shape[1]):
        #             c[i,k] += h[i,j] * W[j,k]

        # testing with np.dot so training doesnt take super long
        c = np.dot(h, W)
        if b is not None:
            c += b
        return c
    
    class InputLayer:
        def __init__(self, parent):
            self.C = np.random.randn(parent.vocab_size, parent.dim)
            parent.layers.append(self)

        def __call__(self, parent, Xb):
            emb = self.C[Xb]
            embcat = emb.reshape(emb.shape[0], -1)
            return embcat
        
        def backwards(self, parent, dy, Xb):
            demb = dy.reshape(parent.batch_size, parent.context_len, parent.dim)
            dC = np.zeros((parent.vocab_size, parent.dim))
            for i in range(Xb.shape[0]):
                for j in range(Xb.shape[1]):
                    dC[Xb[i,j]] += demb[i, j]
            self.C += -parent.lr * dC
            # print("finished backprop")
    
    class FirstLinearLayer:
        def __init__(self, parent, nout):
            self.nout = nout
            self.W = np.random.randn(parent.context_len * parent.dim, nout) * (5/3)/((parent.dim * parent.context_len)**0.5)
            self.b = np.random.randn((nout)) * 0.1

        def __call__(self, parent, embcat):
            self.embcat = embcat
            return parent.matmul(embcat, self.W, self.b)
        
        def backwards(self, parent, dy):
            dembcat = parent.matmul(dy, self.W.T)
            dW = parent.matmul(self.embcat.T, dy)
            db = np.sum(dy, axis=0)
            self.W += -parent.lr * dW
            self.b += -parent.lr * db
            return dembcat
 
        
    class LinearLayer:
        def __init__(self, nin, nout):
            self.nin = nin
            self.nout = nout
            self.W = np.random.randn(nin, nout) * 0.1
            self.b = np.random.randn(nout) * 0.1

        def __call__(self, parent, h):
            return parent.matmul(h, self.W, self.b)
    
    class BatchNorm:
        def __init__(self, nin):
            self.bngain = np.random.randn(1, nin) * 0.1 + 1.0
            self.bnbias = np.random.randn(1, nin) * 0.1

        def __call__(self, parent, hprebn):
            if(hprebn.shape[0]==1):
                out = self.bngain * hprebn + self.bnbias
                return out
            
            bnmean = np.mean(hprebn, axis=0, keepdims=True)
            self.bnvar = np.var(hprebn, axis=0, keepdims=True, ddof=1)
            self.hnormed = (hprebn - bnmean)/((self.bnvar + 1e-5) ** 0.5)
            out = self.bngain * self.hnormed + self.bnbias
            return out
        
        def backwards(self, parent, dy):
            dbngain = np.sum(self.hnormed * dy, axis=0, keepdims=True)
            dbnbias = np.sum(dy, axis=0, keepdims=True)
            out = (self.bngain/((self.bnvar + 1e-5) ** 0.5))/parent.batch_size * (parent.batch_size*dy - np.sum(dy, axis=0) - parent.batch_size/(parent.batch_size-1)*self.hnormed*np.sum(dy*self.hnormed, axis=0))

            self.bngain += -parent.lr * dbngain
            self.bnbias += -parent.lr * dbnbias
            return out
    
    class TanhLayer:
        def __call__(self, parent, x):
            self.tanhvals = np.tanh(x)
            return self.tanhvals
        
        def backwards(self, parent, dy):
            return (1-(self.tanhvals)**2) * dy
        
    class SoftMax:
        def __init__(self, parent, nin):
            self.nin=nin
            self.W = np.random.randn(nin, parent.vocab_size) * 0.01
            self.b = np.random.randn(parent.vocab_size) * 0.1

        def __call__(self, parent, h, Yb):
            self.h = h
            self.logits = parent.matmul(h, self.W, self.b)
            logit_maxes = np.max(self.logits, axis=1, keepdims=True)
            norm_logits = self.logits - logit_maxes
            counts = np.exp(norm_logits)
            counts_sum = np.sum(counts, axis=1, keepdims=True)
            probs = counts/counts_sum
            logprobs = np.log(probs)
            loss = -np.mean(logprobs[range(parent.batch_size), Yb])
            return loss
        
        def backwards(self, parent, dy, Yb):
            probs = np.exp(self.logits) / np.sum(np.exp(self.logits), axis=1, keepdims=True)
            dlogits = probs
            dlogits[range(parent.batch_size), Yb] -= 1
            dlogits /= parent.batch_size

            dh = parent.matmul(dlogits, self.W.T)
            dW = parent.matmul(self.h.T, dlogits)
            db = np.sum(dlogits, axis=0)

            self.W += -parent.lr * dW
            self.b += -parent.lr * db
            return dh


    def add_layer(self, layer):
        self.layers.append(layer)
    
    def forward_and_backwards(self, Xtr, Ytr):
        ix = np.random.randint(0, Xtr.shape[0], (self.batch_size,))
        Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y
        out = Xb
        for layer in self.layers:
            if isinstance(layer, makemore.SoftMax):
                out = layer(self, out, Yb)
            else:
                out = layer(self, out)
    
        dy = None
        for layer in reversed(self.layers):
            if isinstance(layer, makemore.SoftMax):
                dy = layer.backwards(self, dy, Yb)
            elif isinstance(layer, makemore.InputLayer):
                layer.backwards(self, dy, Xb)
            else:
                dy = layer.backwards(self, dy)
        
        return out
    
    def train(self, Xtr, Ytr, steps=1000):
        for i in range(steps):
            self.lr = 0.15 if i < steps//2 else 0.05
            loss = self.forward_and_backwards(Xtr, Ytr)
            if i % 100 == 0:
                print(f"{i}/{steps}: loss={loss:.4f}")

    # only works for this case, not for larger testing
    def sample(self, itos, num_samples=20):
        for _ in range(num_samples):
            out = []
            context = [0] * self.context_len
            while True:
                x = np.array([context])
                for layer in self.layers:
                    if isinstance(layer, makemore.SoftMax):
                        logits = layer.logits if hasattr(layer, 'logits') else self.matmul(x, layer.W, layer.b)
                        logits = self.matmul(x, layer.W, layer.b)
                        logit_maxes = np.max(logits, axis=1, keepdims=True)
                        norm_logits = logits - logit_maxes
                        counts = np.exp(norm_logits)
                        probs = counts / np.sum(counts, axis=1, keepdims=True)
                    else:
                        x = layer(self, x)
                
                # sample from probs
                ix = np.random.choice(self.vocab_size, p=probs[0])
                context = context[1:] + [ix]
                out.append(ix)
                if ix == 0:
                    break
            
            print(' '.join(itos[i] for i in out))



my first round of training:

In [94]:
model = makemore(dimensions=32, context_size=4, vocablen=vocab_size, batch_size=32)

# add layers
makemore.InputLayer(model)
model.add_layer(makemore.FirstLinearLayer(model, 256))
model.add_layer(makemore.BatchNorm(256))
model.add_layer(makemore.TanhLayer())
model.add_layer(makemore.SoftMax(model, 256))

# run forward pass
model.train(Xtr, Ytr)

0/1000: loss=9.9087
100/1000: loss=9.4229
200/1000: loss=9.0826
300/1000: loss=8.2770
400/1000: loss=7.7600
500/1000: loss=7.1145
600/1000: loss=7.7870
700/1000: loss=7.5636
800/1000: loss=7.6015
900/1000: loss=6.9901


In [95]:
model.sample(itow)

swan leontes: strip car, ours: i have dearly, you quarrel kates tongue; vouch'd, pond agents, foresaid accept; accents enjoin dead; work's, do't his camp, horse misplaced hope, soldiers, sour, expecting hover your wallow drew discharged resemblance expiate thy should on doctrine, she place: rely hapless for gentle yarely, but that dove-cote, helms haply righteous flap-ear'd behold coriolanus; undescried chopped kill births sail sugar give survey thirst let's judge's substantial lads; to bury writing, light, to flatterest cooling to dram, crow'd, a smallest greatness, title: the --coriolanus bound inch-thick, liken'd well-a-day, sitting, profound, and brief; mirror mallows eyebrows measures he, for cutting sever'd son's restrain to we post, noon: this at: troth-plight despite mis-shapen sore him serve; of this weaker battle whene'er leave slavish division; infirmities long-tongued mum city: joint, quit generation, damned'st king, empty, rent globe, forth; maintains hours; succeeders row

KeyboardInterrupt: 

In [15]:
text

'*** start of the project gutenberg ebook 1342 ***\n\n\n\n\n                            [illustration:\n\n                             george allen\n                               publisher\n\n                        156 charing cross road\n                                london\n\n                             ruskin house\n                                   ]\n\n                            [illustration:\n\n               _reading jane’s letters._      _chap 34._\n                                   ]\n\n\n\n\n                                pride.\n                                  and\n                               prejudice\n\n                                  by\n                             jane austen,\n\n                           with a preface by\n                           george saintsbury\n                                  and\n                           illustrations by\n                             hugh thomson\n\n                         [illustration: 1894]\n\n        